# Ejercicio 3. Operaciones de una matriz de distancias

**Sistemas de Big Data · Cuaderno resuelto**

Queremos obtener las distancias entre **$n$ objetos**, cada uno descrito por
**$p$ características numéricas**, y contar las operaciones necesarias.

Ejecuta las celdas en orden. Solo se utiliza la biblioteca estándar de Python;
no necesitas archivos externos ni instalar paquetes.

## 1. Distinguir las dos fórmulas

Interpretamos la expresión recibida como la suma de diferencias al cuadrado:

$$d^2(x,y)=\sum_{i=1}^{p}(x_i-y_i)^2$$

Si el enunciado original incluye además la raíz cuadrada, se obtiene la distancia euclídea:

$$d(x,y)=\sqrt{\sum_{i=1}^{p}(x_i-y_i)^2}$$

Este cuaderno permite calcular ambas versiones y cuenta por separado la raíz.
Primero programamos la suma de cuadrados.

In [6]:
from math import sqrt

def distancia_cuadrada(x, y):
    """Calcula la suma de diferencias al cuadrado, con 3*p-1 operaciones.

    Los vectores deben tener la misma longitud positiva. Se supone que sus
    componentes permiten las operaciones numéricas de resta y multiplicación.
    """
    if len(x) != len(y) or len(x) == 0:
        raise ValueError("Los vectores deben tener igual dimensión positiva.")

    # Primer término: una resta y una multiplicación; no se suma a cero.
    diferencia = x[0] - y[0]
    acumulado = diferencia * diferencia

    # Cada una de las p-1 coordenadas restantes requiere una resta,
    # una multiplicación y una suma sobre lo que ya estaba acumulado.
    for i in range(1, len(x)):
        diferencia = x[i] - y[i]
        acumulado += diferencia * diferencia

    return acumulado

## 2. Calcular únicamente las distancias distintas

La matriz tiene $n^2$ posiciones. Sin embargo:

- La distancia de un objeto consigo mismo es cero.
- La distancia de $i$ a $j$ coincide con la de $j$ a $i$.

Por tanto, basta calcular el triángulo situado por encima de la diagonal y copiar
cada resultado a la posición simétrica. El número de distancias calculadas es:

$$\frac{n(n-1)}{2}$$

La función devuelve la matriz completa. No vuelve a calcular la distancia al copiarla.

In [7]:
def matriz_distancias(objetos, con_raiz=False):
    """Devuelve la matriz completa calculando solo los pares con i < j.

    con_raiz=False representa exactamente la fórmula de suma sin raíz.
    con_raiz=True añade la raíz y devuelve distancias euclídeas.
    Una colección vacía produce una matriz vacía.
    """
    n = len(objetos)
    if n == 0:
        return []

    p = len(objetos[0])
    if p == 0 or any(len(objeto) != p for objeto in objetos):
        raise ValueError("Todos los objetos deben tener igual dimensión positiva.")

    # Cada fila es una lista independiente. Los ceros de la diagonal
    # representan la distancia de cada objeto consigo mismo.
    matriz = [[0 for _ in range(n)] for _ in range(n)]

    for i in range(n):
        # j empieza en i+1: se evita la diagonal y el par inverso (j, i).
        for j in range(i + 1, n):
            valor = distancia_cuadrada(objetos[i], objetos[j])
            if con_raiz:
                valor = sqrt(valor)
            matriz[i][j] = valor
            matriz[j][i] = valor

    return matriz

## 3. Contar las operaciones aritméticas

Para una distancia al cuadrado con $p$ características se realizan:

| Operación | Cantidad |
|---|---:|
| Resta de coordenadas | $p$ |
| Multiplicación para obtener los cuadrados | $p$ |
| Suma de los términos | $p-1$ |
| **Total sin raíz** | **$3p-1$** |

La función comienza con el primer término calculado y suma los $p-1$ restantes.
Si comenzase en cero y sumase todos los términos, realizaría una suma más.

Al aprovechar simetría y diagonal nula, el total sin raíz es:

$$N_{\mathrm{op}}=\frac{n(n-1)}{2}(3p-1)$$

Si contamos cada raíz como una operación adicional, el total con raíz es:

$$N_{\mathrm{op,raiz}}=\frac{n(n-1)}{2}(3p)$$

Esta cuenta incluye la aritmética de las distancias, pero excluye control de bucles,
asignaciones y accesos a memoria. No todas esas operaciones tardan lo mismo en un equipo real.

In [8]:
def contar_operaciones(n, p, con_raiz=False):
    """Cuenta distancias únicas y operaciones aritméticas del método anterior."""
    if not isinstance(n, int) or not isinstance(p, int) or n < 0 or p < 1:
        raise ValueError("n debe ser un entero no negativo y p un entero positivo.")

    # El producto n*(n-1) es par; //2 conserva el resultado como entero.
    distancias_unicas = n * (n - 1) // 2
    operaciones_por_distancia = 3 * p - 1
    if con_raiz:
        operaciones_por_distancia += 1

    operaciones_totales = distancias_unicas * operaciones_por_distancia
    return distancias_unicas, operaciones_por_distancia, operaciones_totales

## 4. Ejemplo verificable a mano

Elegimos tres objetos de dos características:

$$A=(0,0),\quad B=(3,4),\quad C=(3,0)$$

Entre $A$ y $B$, la suma de cuadrados vale $(0-3)^2+(0-4)^2=25$.
Su raíz es $5$. Los datos son un ejemplo didáctico: el enunciado no fija $n$ y $p$.

In [9]:
objetos = [[0, 0], [3, 4], [3, 0]]
n = len(objetos)
p = len(objetos[0])

matriz_cuadrada = matriz_distancias(objetos, con_raiz=False)
pares, por_distancia, total = contar_operaciones(n, p, con_raiz=False)

print("Matriz de distancias al cuadrado:")
for fila in matriz_cuadrada:
    print(fila)
print("Distancias distintas calculadas:", pares)
print("Operaciones por distancia:", por_distancia)
print("Operaciones aritméticas totales:", total)

# Se comprueba la matriz con los valores calculables a mano.
assert matriz_cuadrada == [[0, 25, 9], [25, 0, 16], [9, 16, 0]]
assert (pares, por_distancia, total) == (3, 5, 15)

Matriz de distancias al cuadrado:
[0, 25, 9]
[25, 0, 16]
[9, 16, 0]
Distancias distintas calculadas: 3
Operaciones por distancia: 5
Operaciones aritméticas totales: 15


## 5. Variante con raíz cuadrada

Se conservan los mismos pares y se añade una raíz por cada distancia calculada.
En este ejemplo pasan de 15 a 18 las operaciones aritméticas del modelo de cuenta.

In [10]:
matriz_euclidea = matriz_distancias(objetos, con_raiz=True)
pares_raiz, por_distancia_raiz, total_raiz = contar_operaciones(n, p, con_raiz=True)

print("Matriz de distancias euclídeas:")
for fila in matriz_euclidea:
    print(fila)
print("Distancias distintas calculadas:", pares_raiz)
print("Operaciones por distancia:", por_distancia_raiz)
print("Operaciones aritméticas totales:", total_raiz)

assert matriz_euclidea == [[0, 5.0, 3.0], [5.0, 0, 4.0], [3.0, 4.0, 0]]
assert (pares_raiz, por_distancia_raiz, total_raiz) == (3, 6, 18)

Matriz de distancias euclídeas:
[0, 5.0, 3.0]
[5.0, 0, 4.0]
[3.0, 4.0, 0]
Distancias distintas calculadas: 3
Operaciones por distancia: 6
Operaciones aritméticas totales: 18


## 6. Respuesta general y complejidad

| Estrategia | Número de distancias que se calculan |
|---|---:|
| Todas las posiciones, incluida la diagonal | $n^2$ |
| Sin diagonal, calculando también cada par inverso | $n(n-1)$ |
| Aprovechando diagonal nula y simetría | $n(n-1)/2$ |

El tiempo del método general es **$\Theta(n^2p)$**. Con dimensión $p$ constante,
se expresa como **$\Theta(n^2)$**. Almacenar la matriz completa ocupa **$\Theta(n^2)$** posiciones.

No deben confundirse tres cantidades: posiciones de la matriz, distancias distintas
calculadas y operaciones aritméticas de cada cálculo.